# 07 · Show your work 📤

**Generative Modelling for HEP** — open submission portal (teacher setup).

Some students will bring their **own** notebook — a Colab experiment, a wild new
architecture, whatever. This sets up a single **open-submission** checkpoint,
`my-work`, so a student working *anywhere* (Colab, Kaggle, local) can show you what
they made — a few key cells, or the whole notebook.

They have the full "show your work" toolkit, all pointing at `my-work`:
- **A code cell** → `%%cadence_submit my-work` at the top (runs the cell, submits its code)
- **A plot** → `submit_image("my-work", fig)`
- **A written note / markdown cell** → `%%cadence_explain my-work` (text in the cell body)
- **The whole notebook** → one `submit_notebook()` call
- *(optional)* frame it all → `present("title", "notes")` for presenting mode

**You (teacher):** run the two cells below to register the checkpoint under the
`ml4hep-mumbai` course. It uses a **pinned join code — `ml4hep-showcase` — the one code
everyone uses** (already baked into the student instructions below). Everything a
student marks lands on your dashboard under `my-work`, newest first, tagged with
their name.

There are no graded exercises here — it's a submission inbox, not a lesson.

In [2]:
%load_ext cadence
# Optional — uncomment to sign in if this kernel isn't already authenticated:
# %cadence_login --username YOUR_USERNAME

In [3]:
# Register the open-submission checkpoint under the course, then show the join code.
%cadence_course "ml4hep-mumbai"

# First run only — creates the lesson with the PINNED join code below. ⚠️
# %cadence_add_notebook makes a NEW lesson each run, so run this cell ONCE (a second
# run errors on the duplicate code). Re-registering later? Comment the add_notebook
# line and uncomment %cadence_lesson to REUSE this lesson (keeps the same code).
%cadence_add_notebook "07 Submit Your Notebook" --code ml4hep-showcase
# %cadence_lesson "07 Submit Your Notebook"

%cadence_register my-work --comparator manual --allow-submissions
%cadence_show_join

API request failed: 409 Client Error: Conflict for url: https://api.cadence-dash.com/lessons


## Student instructions — share these + the join code

**Everyone first runs this once**, at the top of their own notebook (Colab /
Kaggle / local) — the join code is the same for everyone:

```python
!pip install -q -i https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ cadence-edu==0.3.0
%load_ext cadence
%cadence_session ml4hep-showcase "your name"
```

Then use any of these — mark as much or as little as you like. Everything goes to
the `my-work` checkpoint, so I see it all together under your name.

### Mark the cells you want me to see

Put this at the **top** of a **code** cell. It runs the cell as normal (you still
see the output) and sends its **code** to my dashboard:

```python
%%cadence_submit my-work
# ...your code...
```

Put this at the top of a cell to send a **written note / explanation** (markdown):

```python
%%cadence_explain my-work
Here's what my generator does and where it still struggles...
```

Made a **plot** you want me to actually *see*? Send the figure itself:

```python
submit_image("my-work", fig)     # fig = your matplotlib figure
```

*(Optional)* Give your submission a headline for presenting mode:

```python
from cadence import present
present("My gluon-jet generator", notes="VAE baseline, then a diffusion decoder.")
```

### Or send the whole notebook at once

Prefer to just hand in everything? Paste this at the **end** and run it — it ships
your entire notebook. In Colab it grabs the live notebook automatically; elsewhere
pass `path="your_notebook.ipynb"` (download/save it first). Cell outputs are
stripped to fit a **50 KB** limit (`keep_outputs=True` tries to keep plots but
falls back to source-only if it's too big):

```python
import cadence.progress as _p, json
LIMIT = 50_000   # cadence caps a submission's text at 50 KB
def submit_notebook(name, path=None, keep_outputs=False):
    """Send your WHOLE notebook to the teacher's dashboard.
    In Colab it grabs the live notebook automatically; elsewhere pass
    path="your_notebook.ipynb" (download/save it first). Outputs are stripped so
    it fits the 50 KB limit — pass keep_outputs=True to try to include plots."""
    api, sid = _p._state["api"], _p._state["session_id"]
    if path is None:
        try:
            from google.colab import _message
            nb = _message.blocking_request("get_ipynb", timeout_sec=60)["ipynb"]
        except Exception:
            raise RuntimeError("Not in Colab — pass path='your_notebook.ipynb' "
                               "(download/save your notebook first).")
    else:
        nb = json.load(open(path))
    def dump(strip):
        n = json.loads(json.dumps(nb))                    # work on a copy
        if strip:
            for c in n.get("cells", []):
                c.pop("outputs", None); c.pop("execution_count", None)
        return json.dumps(n, indent=1)
    text = dump(strip=not keep_outputs)
    if len(text) > LIMIT and keep_outputs:                # plots too big — fall back
        text = dump(strip=True)
        print("\u26a0\ufe0f  with outputs it was over the 50 KB limit — sent source-only.")
    if len(text) > LIMIT:
        raise RuntimeError(f"Notebook source is {len(text)//1024} KB, over the 50 KB "
                           "limit — trim it (fewer/shorter cells) or submit key parts.")
    api.submit_code(sid, "my-work", code=text, language="json", note=name)
    print(f"\u2705 submitted '{name}'  ({len(text)//1024} KB)")

submit_notebook("my gluon-jet generator")      # <- title it whatever you like
```

---

Everything lands on your dashboard under **`my-work`** — marked cells as code
snippets, notes as text, plots as images, and the whole-notebook option as one
JSON blob (copy it out, save as `.ipynb`, re-open). `%%cadence_submit` sends
*code*, not the rendered output — that's why plots go through `submit_image`.

> ⚠️ If you regenerate this notebook with `%cadence_autoregister`, note it *won't*
> re-add `--allow-submissions` — the flag lives only on the `%cadence_register`
> line above. Just re-run that cell to re-push it.

In [ ]:
# (optional) quick self-test AS A STUDENT — confirm submissions land.
# Uncomment and run (uses the same pinned code students use).
# %cadence_session ml4hep-showcase "teacher test"
# import cadence.progress as _p, json
# _p._state["api"].submit_code(_p._state["session_id"], "my-work",
#                              code="print('hello from a marked cell')", note="self test")
# print("submitted a test snippet — check your dashboard")